# 第九课｜一个计算单元怎样服务很多神经元？

前面我们一直让一个 RTL neuron 保存自己的 `membrane_v`。但真实系统不会为十万个虚拟神经元各复制一整套计算电路。今天只解决：
> **一个物理计算单元怎样轮流更新很多个虚拟神经元？**

主要新概念：**时间复用（time multiplexing）**。


## 1. 概念账本

**已经知道：** 一个 neuron update 的 combinational path、register state、clocked update。

**今天学习：** time multiplexing；为理解它，会同时认识 memory、address 与 RAM 的基本角色。

**只预告：** spike queue、稀疏突触和 router 留到后面三课。


## 2. 为什么不能简单复制十万份 neuron engine？

复制计算单元可以提高并行度，但会消耗更多逻辑、寄存器和存储资源。另一种办法是把大量 neuron state 放进 memory，只保留较少的计算 engine，让 engine 按顺序读出一个 state、更新、再写回。

这里的重点不是先决定“多少个 engine 最优”，而是理解**物理计算资源数量**和**虚拟神经元数量**可以不同。


## 3. 三个支持术语

**存储器（memory）**：保存许多 state 的地方。

**地址（address）**：指出“这次要读/写 memory 中哪一个位置”的编号。

**随机存取存储器（Random-Access Memory, RAM）**：可以通过 address 选择不同存储位置的 memory。这里先把它看成“很多带编号的 state 槽位”；真实 FPGA 的 BRAM/URAM 细节以后再学。


## 4. time multiplexing 的核心

time multiplexing 的意思是：**同一个物理计算单元，在不同时间片处理不同虚拟对象。**

```mermaid
flowchart LR
 A["address / neuron_id"] --> MEM["state memory"]
 MEM --> ENG["one neuron engine"]
 ENG --> MEM
 SCHED["scheduler: 0,1,2,3,..."] --> A
```

关键不是 Python 的 `for` 循环本身，而是“地址选择 state → 一个 engine 更新 → 写回同一地址”这个架构关系。


## 5. Run：四个虚拟神经元，一个计算过程

下面用 Python list 模拟 state memory。每个 address 只在轮到自己时被更新。先预测最终 memory。


In [ ]:
states = [0, 10, -3, 7]
inputs = [2, -1, 4, 0]

print('address | before | input | after')
for address, input_value in enumerate(inputs):
    before = states[address]
    after = before + input_value
    states[address] = after
    print(f'{address:7d} | {before:6d} | {input_value:5d} | {after:5d}')

print('final memory:', states)


## 6. Observe

观察两件事：

1. `address=0` 更新时，不应该修改其他 address；
2. 一个 pass 完成后，四个 state 都被同一个“更新规则”服务过一次。

这就是后续 `MOD-004 neuron_state_store` 与 scheduler 的最小直觉。


## 7. Try It：资源与时间的交换

假设 8 个 neuron，每个 update 恰好占 1 个 cycle。

- 1 个 engine：完成一轮至少需要多少个 update slot？
- 2 个 engine：理想情况下可以怎样分工？

先回答，再修改 Python 让 address 只处理偶数或奇数，观察两个“虚拟 engine”如何分摊工作。这里先不讨论真实 pipeline latency。


## 8. 作业

完成 `exercises/lesson09_time_multiplexing.py`：

- `update_addressed_state(...)` 只能修改指定 address；
- `round_robin_pass(...)` 必须按 address 顺序服务所有 state。

运行：

```bash
uv run pytest exercises/checks/check_lesson09.py -q
```


## 9. AI Task

让 AI 比较“4 个 neuron 各有一个 engine”和“1 个 engine time-multiplex 4 个 neuron”的资源/时间差异。要求它只做定性比较，不凭空编造 LUT、MHz 或功耗数字。


## 10. Human Check

不用 AI，你应该能解释：address 为什么不是 neuron state；memory 保存什么；一个 engine 如何服务多个虚拟 neuron；为什么 time multiplexing 节省计算资源但通常增加完成一轮更新所需的时间。


## 11. Engineering Handoff

本课只建立 `MOD-004 neuron_state_store` 与 RMD-006/007 的架构直觉，不冻结 RAM 时序、banking、scheduler 或正式 neuron interface。


## 12. 项目追踪 Project Trace

- Lesson: `LSN-009`
- Mapping: `RMD-006 / RMD-007` teaching precursor
- Module context: `MOD-004`
- Formal implementation status: not complete


## 13. Exit Ticket

你能画出 `address → state memory → one engine → write back`，并解释“很多虚拟 neuron”为什么不等于“很多物理 engine”。
